# MAPED on one CUDA GPU

This workflow keeps all seven input tilts in exact encoded form on GPU0. MAPED computes alignment summaries directly from those encoded sources, merges automatically sized regions, and writes a globally scaled uint16 result. The full float32 output is never allocated. The saved result reopens packed for live `Show4DSTEM` viewing. Set `MAPED_DATA_DIR` to the directory containing the seven `*_master.h5` files; optionally set `MAPED_OUTPUT` to choose the saved result path.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from time import perf_counter
import torch
from quantem.diffraction import MAPEDTorch

try:
    SESSION = Path(os.environ["MAPED_DATA_DIR"]).expanduser().resolve()
except KeyError as exc:
    raise RuntimeError("Set MAPED_DATA_DIR to the directory containing seven *_master.h5 files.") from exc
FILES = sorted(SESSION.glob("*_master.h5"))
if len(FILES) != 7:
    raise ValueError(f"Expected seven *_master.h5 files in {SESSION}, found {len(FILES)}.")
OUTPUT = Path(os.environ.get("MAPED_OUTPUT", "maped-output/merged_master.h5")).expanduser().resolve()
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print(torch.cuda.get_device_name(0), "|", len(FILES), "tilts")


In [ ]:
started = perf_counter()
maped = MAPEDTorch.from_files(FILES, device="cuda:0")
torch.cuda.synchronize()
sources = maped.datasets.sources
assert len(sources) == len(FILES) == 7
assert all(source.representation.value == "encoded" for source in sources)
assert all(source.metadata["source_read_passes"] == 1 for source in sources)
print(f"Encoded resident ready: {sum(x.resident_bytes for x in sources) / 2**30:.2f} GiB in {perf_counter() - started:.2f} s")


In [ ]:
started = perf_counter()
maped.preprocess(plot_summary=False)
maped.diffraction_origin(sigma=1, plot_origins=False)
maped.diffraction_align(edge_blend=2, plot_aligned=False)
maped.real_space_align(
    num_iter=20, hanning_filter=True, padding=2, edge_blend=5,
    pad_val="median", shift_method="bilinear", plot_aligned=False,
)
torch.cuda.synchronize()
print(f"Alignment: {perf_counter() - started:.2f} s")

In [ ]:
started = perf_counter()
merged = maped.merge_datasets(save_to=OUTPUT, plot_result=False)
torch.cuda.synchronize()
precision = merged.metadata["precision"]
print(f"Merge + save + reopen: {perf_counter() - started:.2f} s")
print(f"scaled uint16 | RMSE {precision['rmse']:.4g} | max error {precision['max_abs_error']:.4g} | clipped {precision['clipped']}")

In [ ]:
viewer = maped.show()
viewer

The seven encoded inputs and packed merged result remain resident for responsive inspection. After the final viewer consumer is closed, release MAPED-owned GPU resources with:

```python
maped.close()
```


## Latest validated CUDA GPU0 run

The isolated reference workflow finished in **31.39 s** with an **11.15 GiB** MAPED process peak (**12.08 GiB total GPU0**). Inputs occupied **7.76 GiB** in encoded form and each HDF5 source was read once. GPU scaled-uint16 encoding took **0.176 s**; packed reopen took **2.53 s**; `Show4DSTEM` construction took **0.249 s**. The output reported RMSE **0.006951**, maximum absolute error **0.01318**, and **0 clipped values**.

The current synced worktree also completed a local notebook smoke run: encoded loading **10.21 s**, alignment **2.03 s**, and merge/save/reopen **30.90 s**. Budget about **43–45 s end to end** for this notebook run; isolated GPU benchmarks are typically **31–36 s**.

![Three diffraction patterns from the packed merged output](selected_diffraction_patterns.png)
